In [13]:
import pandas as pd
import geopandas as gpd
import numpy as np
import h3
import os
import joblib

print("Step 1: Loading Tabular and Spatial Data...")

# 1. Load the Uber CSV Travel Times
df = pd.read_csv('../data/uber_data.csv')
columns_to_keep = ['sourceid', 'dstid', 'hod', 'mean_travel_time']
df = df[columns_to_keep].dropna()

# 2. Load the JSON Map Boundaries
# geopandas reads the raw coordinates and turns them into mathematical Polygon objects
gdf = gpd.read_file('../data/Bangalore Wards 2016-2018.json')

print(f"Loaded {len(df)} trips and {len(gdf)} map zones.")
print("JSON properties available:", gdf.columns.tolist())

Step 1: Loading Tabular and Spatial Data...
Loaded 825319 trips and 198 map zones.
JSON properties available: ['WARD_NO', 'WARD_NAME', 'MOVEMENT_ID', 'DISPLAY_NAME', 'geometry']


In [14]:
print("Step 2: H3 Spatial Engineering...")

# A. Find the center of every Zone Polygon
# We calculate the mathematical centroid (lat/lon) of each ward boundary
centroids = gdf.geometry.centroid
gdf['lat'] = centroids.y
# Sometimes JSON maps have flipped coordinates, so we handle both safely
gdf['lon'] = centroids.x 

# B. Convert the GPS Centers to Uber H3 Hexagons (Resolution 9)
H3_RES = 9

# Handle different versions of the h3 library (v3 vs v4)
try:
    gdf['hex_id'] = gdf.apply(lambda row: h3.latlng_to_cell(row.lat, row.lon, H3_RES), axis=1)
except AttributeError:
    gdf['hex_id'] = gdf.apply(lambda row: h3.geo_to_h3(row.lat, row.lon, H3_RES), axis=1)

# C. Create a Translation Dictionary
# The JSON file usually stores the Zone ID in a column called 'MOVEMENT_ID' or 'id'.
# We map that ID to our new H3 Hexagon.
zone_id_col = 'MOVEMENT_ID' if 'MOVEMENT_ID' in gdf.columns else 'MOVEMENT_I' if 'MOVEMENT_I' in gdf.columns else 'id' 
zone_to_hex_dict = dict(zip(gdf[zone_id_col].astype(int), gdf['hex_id']))

# D. Map the Hexagons into our Uber Training Data
df['source_hex'] = df['sourceid'].map(zone_to_hex_dict)
df['dest_hex'] = df['dstid'].map(zone_to_hex_dict)

# Drop trips where the JSON didn't have a matching boundary
df = df.dropna(subset=['source_hex', 'dest_hex'])

# E. Target Encoding based on HEXAGONS, not Zones
source_hex_mean = df.groupby('source_hex')['mean_travel_time'].mean()
dest_hex_mean = df.groupby('dest_hex')['mean_travel_time'].mean()

df['source_avg_time'] = df['source_hex'].map(source_hex_mean)
df['dest_avg_time'] = df['dest_hex'].map(dest_hex_mean)

# Save the Hexagon Encoders for our FastAPI backend
os.makedirs('../models', exist_ok=True)
joblib.dump(source_hex_mean, '../models/source_hex_encoding.pkl')
joblib.dump(dest_hex_mean, '../models/dest_hex_encoding.pkl')

# F. Cyclical Time Engineering
df['hod_sin'] = np.sin(2 * np.pi * df['hod'] / 24)
df['hod_cos'] = np.cos(2 * np.pi * df['hod'] / 24)

# Final cleanup before training
X = df.drop(columns=['mean_travel_time', 'sourceid', 'dstid', 'hod', 'source_hex', 'dest_hex'])
y = df['mean_travel_time']

print("Success! Advanced H3 Features Engineered:", X.columns.tolist())

Step 2: H3 Spatial Engineering...
Success! Advanced H3 Features Engineered: ['source_avg_time', 'dest_avg_time', 'hod_sin', 'hod_cos']


/var/folders/f5/gphj8y9d4mzf_dy_chxvkj7h0000gn/T/ipykernel_64253/1944712567.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


In [15]:
print("Step 3: Train/Test Split...")
# We hold back 20% of the data to test if the models actually learned the traffic patterns
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Testing set: {X_test.shape[0]:,} samples")

Step 3: Train/Test Split...
Training set: 660,255 samples
Testing set: 165,064 samples


In [16]:
print("Step 4: Training Ensemble Models...")

# 1. Train XGBoost (Excellent at finding complex, non-linear patterns)
print("-> Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=150, max_depth=7, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)

# 2. Train LightGBM (Highly optimized and fast for tabular data)
print("-> Training LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=150, max_depth=7, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)

# 3. Train Random Forest (Acts as a stable baseline to prevent overfitting)
print("-> Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("All models successfully trained!")

Step 4: Training Ensemble Models...
-> Training XGBoost...
-> Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000953 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 438
[LightGBM] [Info] Number of data points in the train set: 660255, number of used features: 4
[LightGBM] [Info] Start training from score 1999.457240
-> Training Random Forest...
All models successfully trained!


In [17]:
print("Step 5: Model Evaluation & Exporting...")

def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"\n--- {name} ---")
    print(f"Mean Absolute Error (MAE):  {mae:.2f} seconds")
    print(f"Root Mean Squared Error:    {rmse:.2f} seconds")
    print(f"Accuracy Score (R2):        {r2:.4f}")

# Check how well each model performs on the 20% of data it has never seen
evaluate_model("XGBoost", xgb_model, X_test, y_test)
evaluate_model("LightGBM", lgb_model, X_test, y_test)
evaluate_model("Random Forest", rf_model, X_test, y_test)

# Export the trained "Brains" so your matcher.py server can use them
joblib.dump(xgb_model, '../models/xgb_model.pkl')
joblib.dump(lgb_model, '../models/lgb_model.pkl')
joblib.dump(rf_model, '../models/rf_model.pkl')

print("\nPipeline Complete! All models saved to the /models directory.")

Step 5: Model Evaluation & Exporting...

--- XGBoost ---
Mean Absolute Error (MAE):  568.97 seconds
Root Mean Squared Error:    735.29 seconds
Accuracy Score (R2):        0.5353

--- LightGBM ---
Mean Absolute Error (MAE):  580.58 seconds
Root Mean Squared Error:    747.56 seconds
Accuracy Score (R2):        0.5196

--- Random Forest ---
Mean Absolute Error (MAE):  562.45 seconds
Root Mean Squared Error:    726.60 seconds
Accuracy Score (R2):        0.5462

Pipeline Complete! All models saved to the /models directory.
